<a href="https://colab.research.google.com/github/Manjot-Coder/IP_LabAssignments/blob/main/Assignment-8/All%20questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#ALL QUESTIONS


import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
import io

# ------------------------------------------
# STEP 1: Upload grayscale image
# ------------------------------------------

uploaded = files.upload()
filename = next(iter(uploaded))

img = Image.open(io.BytesIO(uploaded[filename])).convert("L")
image = np.array(img, dtype=float)

rng = np.random.default_rng()

# ------------------------------------------
# STEP 2: Take parameters from user
# ------------------------------------------

print("Enter noise parameters")

u_min = float(input("Uniform minimum (e.g. -30): "))
u_max = float(input("Uniform maximum (e.g. 30): "))

mu = float(input("Gaussian mean (e.g. 0): "))
sigma = float(input("Gaussian standard deviation (e.g. 20): "))

k = int(input("Erlang shape parameter k (e.g. 3): "))
rate = float(input("Erlang rate parameter (e.g. 0.1): "))

exp_rate = float(input("Exponential rate parameter (e.g. 0.05): "))

rayleigh_b = float(input("Rayleigh parameter b (e.g. 1000): "))

if u_max <= u_min or sigma <= 0 or k <= 0 or rate <= 0 or exp_rate <= 0 or rayleigh_b <= 0:
    raise ValueError("Invalid noise parameters.")


# ------------------------------------------
# STEP 3: Generate noise using equations
# ------------------------------------------

shape = image.shape
N = image.size

# 1. Uniform Noise
U = rng.random(shape)

uniform_noise = u_min + (u_max - u_min) * U


# 2. Gaussian Noise (Box-Muller method)
U1 = np.maximum(rng.random(shape), 1e-10)
U2 = rng.random(shape)

gaussian_noise = (
    mu + sigma * np.sqrt(-2 * np.log(U1)) * np.cos(2 * np.pi * U2)
)


# 3. Erlang Noise
# Sum of k independent exponential random variables

erlang_noise = np.zeros(shape)

for i in range(k):
    U = np.maximum(rng.random(shape), 1e-10)
    erlang_noise += -np.log(U) / rate

# Make it zero-mean for additive image noise
erlang_noise -= k / rate


# 4. Exponential Noise

U = np.maximum(rng.random(shape), 1e-10)

exponential_noise = -np.log(U) / exp_rate

# Make it zero-mean
exponential_noise -= 1 / exp_rate


# 5. Rayleigh Noise

U = np.maximum(rng.random(shape), 1e-10)

rayleigh_noise = np.sqrt(-rayleigh_b * np.log(1 - U))

# Make it zero-mean
rayleigh_mean = np.sqrt(rayleigh_b) * np.sqrt(np.pi) / 2

rayleigh_noise -= rayleigh_mean


# ------------------------------------------
# STEP 4: Add noise to original image
# ------------------------------------------

noises = [
    uniform_noise,
    gaussian_noise,
    erlang_noise,
    exponential_noise,
    rayleigh_noise
]

noise_names = [
    "Uniform Noise",
    "Gaussian Noise",
    "Erlang Noise",
    "Exponential Noise",
    "Rayleigh Noise"
]

noisy_images = []

for noise in noises:
    noisy = np.clip(image + noise, 0, 255)
    noisy_images.append(noisy.astype(np.uint8))


# ------------------------------------------
# STEP 5: Generate reference noise using
# NumPy built-in random functions
# ------------------------------------------

builtin_noises = [
    rng.uniform(u_min, u_max, shape),

    rng.normal(mu, sigma, shape),

    rng.gamma(k, 1 / rate, shape) - k / rate,

    rng.exponential(1 / exp_rate, shape) - 1 / exp_rate,

    rng.rayleigh(np.sqrt(rayleigh_b / 2), shape) - rayleigh_mean
]


# ------------------------------------------
# STEP 6: Display noisy images
# ------------------------------------------

plt.figure(figsize=(15, 9))

plt.subplot(2, 3, 1)
plt.imshow(image, cmap="gray")
plt.title("Original Image")
plt.axis("off")

for i in range(5):
    plt.subplot(2, 3, i + 2)
    plt.imshow(noisy_images[i], cmap="gray")
    plt.title(noise_names[i])
    plt.axis("off")

plt.tight_layout()
plt.show()


# ------------------------------------------
# STEP 7: Plot histograms of generated noise
# Compare equation-based and built-in noise
# ------------------------------------------

plt.figure(figsize=(15, 12))

for i in range(5):

    plt.subplot(3, 2, i + 1)

    plt.hist(
        noises[i].flatten(),
        bins=100,
        density=True,
        alpha=0.6,
        label="Equation Based"
    )

    plt.hist(
        builtin_noises[i].flatten(),
        bins=100,
        density=True,
        histtype="step",
        linewidth=2,
        label="Built-in Reference"
    )

    plt.title(noise_names[i])
    plt.xlabel("Noise Intensity")
    plt.ylabel("Probability Density")
    plt.legend()

plt.tight_layout()
plt.show()


# ------------------------------------------
# STEP 8: Save noisy images
# ------------------------------------------

for i in range(5):
    Image.fromarray(noisy_images[i]).save(
        noise_names[i].lower().replace(" ", "_") + ".jpg"
    )

print("All noisy images saved successfully!")
